In [1]:
import os
import random
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")

In [2]:
project_dir = Path.cwd()
DATA_PATH = project_dir / "data" / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst rows:")
display(df.head())

# Clean column names and keep only usable features.
df.columns = df.columns.str.strip()
metadata_cols = ["Flow ID", "Source IP", "Destination IP", "Timestamp"]
metadata_cols = [col for col in metadata_cols if col in df.columns]
df = df.drop(columns=metadata_cols)

df = df.replace([np.inf, -np.inf], np.nan)

for col in df.columns:
    if col == "Label":
        continue
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Replace impossible values and missing data with safe medians.
for col in df.columns:
    if col == "Label":
        continue
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    median = df[col].median()
    if pd.isna(median):
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna(median)

# Label encoding.
df["Label"] = df["Label"].astype(str).str.strip()
df["Label"] = df["Label"].map({"BENIGN": 0, "DDoS": 1})
print("\nLabel distribution:")
print(df["Label"].value_counts().sort_index())

X = df.drop(columns=["Label"])
y = df["Label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

models = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SEED,
                    class_weight="balanced",
                ),
            ),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "MLP": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32),
                    max_iter=500,
                    random_state=SEED,
                ),
            ),
        ]
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    result = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }

    if hasattr(model, "predict_proba"):
        try:
            result["roc_auc"] = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        except Exception:
            pass

    results[name] = result

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["BENIGN", "DDoS"]))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

summary = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\nSummary of model performance:")
print(summary)

rf_model = models["Random Forest"]
feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features:")
print(feature_importance.head(10))

fig_dir = project_dir / "figures"
fig_dir.mkdir(exist_ok=True)

feature_importance.head(15).sort_values().plot(kind="barh", figsize=(10, 6), color="steelblue")
plt.title("Top 15 features for DDoS detection")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(fig_dir / "top_features.png", dpi=150)
plt.close()

results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)
summary.to_csv(results_dir / "model_metrics.csv")
print("\nSaved metrics to:", results_dir / "model_metrics.csv")
print("Saved feature plot to:", fig_dir / "top_features.png")

Dataset shape: (225745, 85)

Column names:
['Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' 

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443,192.168.10.5,54865,6,7/7/2017 3:30,3,2,0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80,192.168.10.5,55054,6,7/7/2017 3:30,109,1,1,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80,192.168.10.5,55055,6,7/7/2017 3:30,52,1,1,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443,192.168.10.16,46236,6,7/7/2017 3:30,34,1,1,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443,192.168.10.5,54863,6,7/7/2017 3:30,3,2,0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN



Label distribution:
Label
0     97718
1    128027
Name: count, dtype: int64



=== Logistic Regression ===
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19544
        DDoS       1.00      1.00      1.00     25605

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149

Confusion matrix:
 [[19513    31]
 [   26 25579]]



=== Random Forest ===
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19544
        DDoS       1.00      1.00      1.00     25605

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149

Confusion matrix:
 [[19543     1]
 [    3 25602]]



=== MLP ===
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19544
        DDoS       1.00      1.00      1.00     25605

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149

Confusion matrix:
 [[19537     7]
 [   11 25594]]

Summary of model performance:
                     accuracy  precision    recall        f1   roc_auc
Random Forest        0.999911   0.999961  0.999883  0.999922  1.000000
MLP                  0.999601   0.999727  0.999570  0.999648  0.999978
Logistic Regression  0.998738   0.998790  0.998985  0.998887  0.999898

Top 10 features:
Fwd Packet Length Mean         0.081717
Fwd Packet Length Max          0.076218
Avg Fwd Segment Size           0.061476
Subflow Fwd Bytes              0.061187
act_data_pkt_fwd               0.054439
Init_Win_bytes_forward         0.044750
Total Length of Fwd Packets    0.0446


Saved metrics to: /Users/nishantgandhi/Library/CloudStorage/OneDrive-LuleåUniversityofTechnology/D7084E AI For Cybersecurity XAI/Lab1/results/model_metrics.csv
Saved feature plot to: /Users/nishantgandhi/Library/CloudStorage/OneDrive-LuleåUniversityofTechnology/D7084E AI For Cybersecurity XAI/Lab1/figures/top_features.png
